In [ ]:
# Verified October 30 sample; it predates the newer steps/<step-name>/ layout.
# The parameters URI is tied to a source release, independently of RUN_ID.
MODEL_ID = "usecase2"
RUN_ID = "cd97jkwzjyqo"

S3_INPUTS = {
    "data": f"s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/{MODEL_ID}/runs/{RUN_ID}/data/raw/data.csv",
    "selection": f"s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/{MODEL_ID}/runs/{RUN_ID}/data/selection/selected_features.csv",
    "parameters": f"s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/source/direct-deploy/my-review/sha256-ceaa55b820e76bdbd411b3809b29094cdcb2b68429e46da84f1b3b4705932745/mlops_models/models/{MODEL_ID}/parameters.json",
}


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import sys

import boto3
import pandas as pd
from botocore.config import Config as ClientConfig
from IPython.display import display

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))

work_directory = Path(mkdtemp(prefix="preprocess-training-"))
data_directory = work_directory / "raw"
selection_file = work_directory / "selection/selected_features.csv"
parameter_path = work_directory / "settings/parameters.json"
output_directory = work_directory / "output"
output_directory.mkdir()
local_files = {
    "data": data_directory / "data.csv",
    "selection": selection_file,
    "parameters": parameter_path,
}
s3 = boto3.client("s3", config=ClientConfig(
    connect_timeout=5, read_timeout=15, retries={"total_max_attempts": 2}))
for name, local_file in local_files.items():
    address = urlsplit(S3_INPUTS[name])
    if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
        raise ValueError(f"Set S3_INPUTS[{name!r}] to an S3 object path in the first cell.")
    local_file.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {S3_INPUTS[name]} -> {local_file}", flush=True)
    s3.download_file(address.netloc, address.path.lstrip("/"), str(local_file))
    assert local_file.is_file(), f"Input was not downloaded: {local_file}"
print("Local output directory:", output_directory)


In [ ]:
# Optional inspection. Skip this cell to go straight to process_train().
display(json.loads(parameter_path.read_text()))
display(pd.read_csv(selection_file))
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))


In [ ]:
from mlops_models.model_base import preprocess

expected_source = REPOSITORY / "mlops-models/src/mlops_models/model_base/preprocess.py"
assert Path(preprocess.__file__).resolve() == expected_source.resolve(), "Restart the kernel in this checkout."
preprocess = importlib.reload(preprocess)
print("Preprocessing source:", preprocess.__file__)

# The current base implementation ignores config. Rerun this cell after code edits.
result_directory = preprocess.process_train(
    data_directory=data_directory,
    selection_file=selection_file,
    output_directory=output_directory,
    config={},
    parameter_path=parameter_path,
)
print("Prepared CSV:", result_directory / "data.csv")
result_directory
